# Bài 2: Dry Bean Dataset

In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


In [48]:
TRAIN_DATA = "Homework_b7/Dry_Bean_Dataset/dry_bean_train.csv"
TEST_DATA = "Homework_b7/Dry_Bean_Dataset/dry_bean_test.csv"

df_train = pd.read_csv(TRAIN_DATA)
df_test = pd.read_csv(TEST_DATA)

print("Kích thước tập train:", df_train.shape)
print("Kích thước tập test:", df_test.shape)


Kích thước tập train: (10834, 17)
Kích thước tập test: (2709, 17)


In [49]:
#Xử lí missing
train_missing = df_train.isnull().sum()
train_missing_percent = (df_train.isnull().sum() / len(df_train)) * 100
train_missing_df = pd.DataFrame({
    'Số lượng (Count)': train_missing,
    'Phần trăm (%)': train_missing_percent
})


test_missing = df_test.isnull().sum()
test_missing_percent = (df_test.isnull().sum() / len(df_test)) * 100
test_missing_df = pd.DataFrame({
    'Số lượng (Count)': test_missing,
    'Phần trăm (%)': test_missing_percent
})

print("Missing tập train:")
display(train_missing_df)

print("Missing tập test:")
display(test_missing_df)

Missing tập train:


,Số lượng (Count),Phần trăm (%)
area,0,0.0
perimeter,0,0.0
majoraxislength,0,0.0
minoraxislength,0,0.0
aspectration,0,0.0
eccentricity,0,0.0
convexarea,0,0.0
equivdiameter,0,0.0
extent,0,0.0
solidity,0,0.0


Missing tập test:


,Số lượng (Count),Phần trăm (%)
area,0,0.0
perimeter,0,0.0
majoraxislength,0,0.0
minoraxislength,0,0.0
aspectration,0,0.0
eccentricity,0,0.0
convexarea,0,0.0
equivdiameter,0,0.0
extent,0,0.0
solidity,0,0.0


In [50]:
#Scale dữ liệu
X_train = df_train.drop(columns=['class'])
y_train = df_train['class']

X_test = df_test.drop(columns=['class'])
y_test = df_test['class']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [51]:
df_train = pd.read_csv(TRAIN_DATA)
df_test = pd.read_csv(TEST_DATA)
le = LabelEncoder()
df_train['class'] = le.fit_transform(df_train['class'])
df_test['class'] = le.transform(df_test['class'])
print("Danh sách các loại đậu:", le.classes_)
mapping_df = pd.DataFrame({
    'Mã số': range(len(le.classes_)),
    'Tên loại đậu': le.classes_
})

display(mapping_df)

Danh sách các loại đậu: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']


,Mã số,Tên loại đậu
0,0,BARBUNYA
1,1,BOMBAY
2,2,CALI
3,3,DERMASON
4,4,HOROZ
5,5,SEKER
6,6,SIRA


# Logistic Regression

In [52]:
logistic_model = LogisticRegression(max_iter=2000)
logistic_model.fit(X_train_scaled, y_train)

def run_and_evaluate_log(X_test, y_test):
    y_pred = logistic_model.predict(X_test)
    Accuracy = accuracy_score(y_test, y_pred)
    F1_score = f1_score(y_test, y_pred, average='weighted')
    Precision = precision_score(y_test, y_pred, average='weighted')
    Recall = recall_score(y_test, y_pred, average='weighted')
    print(f"Logistic Regression - Accuracy: {Accuracy:.4f}, F1_score: {F1_score:.4f}, Precision: {Precision:.4f}, Recall: {Recall:.4f}")
    return None

run_and_evaluate_log(X_test_scaled, y_test)

Logistic Regression - Accuracy: 0.9192, F1_score: 0.9193, Precision: 0.9197, Recall: 0.9192


# K-Nearest Neighbors

In [53]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
def run_and_evaluate_knn(X_test, y_test):
    y_pred = knn_model.predict(X_test)
    
    Accuracy = accuracy_score(y_test, y_pred)
    F1_score = f1_score(y_test, y_pred, average='weighted')
    Precision = precision_score(y_test, y_pred, average='weighted')
    Recall = recall_score(y_test, y_pred, average='weighted')
    
    print(f"KNN (K=5) - Accuracy: {Accuracy:.4f}, F1_score: {F1_score:.4f}, Precision: {Precision:.4f}, Recall: {Recall:.4f}")
    return None

run_and_evaluate_knn(X_test_scaled, y_test)

KNN (K=5) - Accuracy: 0.9155, F1_score: 0.9157, Precision: 0.9163, Recall: 0.9155


In [54]:
y_pred_log = logistic_model.predict(X_test_scaled)
y_pred_knn = knn_model.predict(X_test_scaled)
comparison_df = pd.DataFrame({
    'Thực tế (y_test)': y_test.iloc[:50].values,
    'Logistic Regression': y_pred_log[:50],
    'KNN': y_pred_knn[:50]
})
missed_predict = (comparison_df['Thực tế (y_test)'] != comparison_df['Logistic Regression']).sum()
missed_predict_knn = (comparison_df['Thực tế (y_test)'] != comparison_df['KNN']).sum()

print("Bảng dự đoán mô hình Logistic và KNN:")
display(comparison_df)
print("Logistic:")
print(f"Số lượng dự đoán ĐÚNG: {len(comparison_df) - missed_predict}")
print(f"Số lượng dự đoán SAI: {missed_predict}")

print("\nKNN:")
print(f"Số lượng dự đoán ĐÚNG: {len(comparison_df) - missed_predict_knn}")
print(f"Số lượng dự đoán SAI: {missed_predict_knn}")

Bảng dự đoán mô hình Logistic và KNN:


,Thực tế (y_test),Logistic Regression,KNN
0,SEKER,SEKER,SEKER
1,DERMASON,DERMASON,DERMASON
2,DERMASON,DERMASON,DERMASON
3,HOROZ,HOROZ,HOROZ
4,SEKER,SEKER,SEKER
5,DERMASON,DERMASON,DERMASON
6,DERMASON,DERMASON,DERMASON
7,DERMASON,DERMASON,DERMASON
8,BARBUNYA,BARBUNYA,BARBUNYA
9,DERMASON,DERMASON,DERMASON


Logistic:
Số lượng dự đoán ĐÚNG: 49
Số lượng dự đoán SAI: 1

KNN:
Số lượng dự đoán ĐÚNG: 49
Số lượng dự đoán SAI: 1


In [55]:
run_and_evaluate_log(X_test_scaled, y_test)
run_and_evaluate_knn(X_test_scaled, y_test)

Logistic Regression - Accuracy: 0.9192, F1_score: 0.9193, Precision: 0.9197, Recall: 0.9192
KNN (K=5) - Accuracy: 0.9155, F1_score: 0.9157, Precision: 0.9163, Recall: 0.9155


Nhận xét:
- Cả 2 mô hình đều cho ra hiệu năng rất cao, đều đạt độ chính xác xấp xỉ 92% trên tập test
- Các chỉ số Accuracy, F1-score, Precision, Recall của từng mô hình gần như bằng nhau. Điều này cho thấy mô hình dự đoán tốt và đồng đều trên cả 7 loại đậu chứ không bị tình trạng chỉ đoán đúng 1 vài lớp phổ biến.
- Logistic Regression nhỉnh hơn KNN khoảng 0.37% ở tất cả các chỉ số (khá nhỏ).

Như vậy, với bài toán Dry Bean ta có thể dùng Logistic và KNN đều cho ra kết quả tốt, tuy nhiên Logistic có phần nhỉnh hơn và tốc độ dự đoán nhanh hơn, tiết kiệm bộ nhớ hơn.